# IPF — sealed Q3 2026 holdout

Run this notebook in **QuantConnect Research**, not as a backtest. It uses the pre-registered frozen P90/09:31 rule and first checks whether Research can access the withheld July–September 2026 minute data.

Do not edit thresholds, dates, or timestamps.

In [ ]:
from AlgorithmImports import *
import pandas as pd
import numpy as np
from datetime import datetime
from decimal import Decimal, ROUND_HALF_UP

# Frozen sealed-Q3 constants
P90 = 1.7796
P95 = 2.3646
SIGNAL_START = pd.Timestamp("2026-07-02")
SIGNAL_END = pd.Timestamp("2026-09-29")
START_EQUITY = 2000.0
MARGIN_RATE = 0.05
BASE_SPREAD_POINTS = 1.0

qb = QuantBook()
qb.set_time_zone(TimeZones.NEW_YORK)

qqq = qb.add_equity("QQQ", Resolution.MINUTE).symbol
ndx = qb.add_index("NDX", Resolution.MINUTE).symbol

# Includes enough prior history for a fully populated 252-session prior-only z-score.
start = datetime(2025, 1, 2)
end = datetime(2026, 10, 1)

h_q = qb.history(qqq, start, end, Resolution.MINUTE)
h_n = qb.history(ndx, start, end, Resolution.MINUTE)

def normalize_history(h):
    if h is None or len(h) == 0:
        return pd.DataFrame(columns=["time", "close"])
    df = h.copy().reset_index()
    df.columns = [str(c).lower() for c in df.columns]
    if "time" not in df.columns:
        # QuantConnect normally exposes 'time'; this catches alternative index naming.
        candidates = [c for c in df.columns if "time" in c or "date" in c]
        if not candidates:
            raise RuntimeError(f"Could not identify time column. Columns={list(df.columns)}")
        df = df.rename(columns={candidates[0]: "time"})
    if "close" not in df.columns:
        raise RuntimeError(f"Could not identify close column. Columns={list(df.columns)}")
    df["time"] = pd.to_datetime(df["time"])
    # QuantBook uses the configured NY timezone. If pandas returns an aware index,
    # explicitly convert it to NY before removing tz info for simple clock matching.
    try:
        if df["time"].dt.tz is not None:
            df["time"] = df["time"].dt.tz_convert("America/New_York").dt.tz_localize(None)
    except Exception:
        pass
    return df[["time", "close"]].sort_values("time").reset_index(drop=True)

q = normalize_history(h_q)
n = normalize_history(h_n)

print(
    "QC_RESEARCH_ACCESS,"
    f"qqq_rows={len(q)},ndx_rows={len(n)},"
    f"qqq_max={q['time'].max() if len(q) else None},"
    f"ndx_max={n['time'].max() if len(n) else None}"
)

if q.empty or n.empty:
    raise RuntimeError("QC Research returned no minute history.")

# Hard gate: do not silently run a partial Q3 holdout.
if q["time"].max() < pd.Timestamp("2026-09-30 16:00:00") or n["time"].max() < pd.Timestamp("2026-09-30 10:00:00"):
    raise RuntimeError(
        "STOP_NO_CURRENT_DATA: QuantConnect Research also cannot see the sealed Q3 window. "
        f"QQQ max={q['time'].max()} NDX max={n['time'].max()}"
    )

# Build one row per QQQ regular session from exact 15:45 and 16:00 NY bars.
q["date"] = q["time"].dt.normalize()

q1545 = (
    q[(q["time"].dt.hour == 15) & (q["time"].dt.minute == 45)]
    [["date", "close"]]
    .drop_duplicates("date", keep="last")
    .rename(columns={"close": "px1545"})
)
q1600 = (
    q[(q["time"].dt.hour == 16) & (q["time"].dt.minute == 0)]
    [["date", "close"]]
    .drop_duplicates("date", keep="last")
    .rename(columns={"close": "close1600"})
)

daily = q1545.merge(q1600, on="date", how="inner").sort_values("date").reset_index(drop=True)
daily["prev_close"] = daily["close1600"].shift(1)
daily["signal_return"] = daily["px1545"] / daily["prev_close"] - 1.0

# Frozen prior-only z: today's observation is excluded from its own mean/sd.
s = daily["signal_return"]
mu = s.shift(1).rolling(252, min_periods=60).mean()
sd = s.shift(1).rolling(252, min_periods=60).std()
daily["z"] = (s - mu) / sd
daily["abs_z"] = daily["z"].abs()

# Exact NDX execution bars by session date.
n["date"] = n["time"].dt.normalize()
n0931 = (
    n[(n["time"].dt.hour == 9) & (n["time"].dt.minute == 31)]
    [["date", "close"]]
    .drop_duplicates("date", keep="last")
    .rename(columns={"close": "entry"})
)
n1000 = (
    n[(n["time"].dt.hour == 10) & (n["time"].dt.minute == 0)]
    [["date", "close"]]
    .drop_duplicates("date", keep="last")
    .rename(columns={"close": "exit"})
)
ndaily = n0931.merge(n1000, on="date", how="inner").sort_values("date").reset_index(drop=True)
trade_dates = list(ndaily["date"])

def next_trade_date(signal_date):
    later = [d for d in trade_dates if d > signal_date]
    return later[0] if later else pd.NaT

events = daily[
    (daily["date"] >= SIGNAL_START) &
    (daily["date"] <= SIGNAL_END) &
    (daily["abs_z"] >= P90)
].copy()

events["trade_date"] = events["date"].apply(next_trade_date)
events = events.merge(ndaily, left_on="trade_date", right_on="date", how="left", suffixes=("_signal", "_trade"))
events = events.drop(columns=["date_trade"]).rename(columns={"date_signal": "signal_date"})
events["p95"] = events["abs_z"] >= P95
events["signal_direction"] = np.where(events["signal_return"] > 0, "UP", "DOWN")
events["fade_sign"] = np.where(events["signal_return"] > 0, -1.0, 1.0)
events["gross_points"] = events["fade_sign"] * (events["exit"] - events["entry"])
events["gross_bp"] = events["gross_points"] / events["entry"] * 10000.0
events["net_points"] = events["gross_points"] - BASE_SPREAD_POINTS
events["net_bp"] = events["gross_bp"] - BASE_SPREAD_POINTS / events["entry"] * 10000.0
events["net_points_2x"] = events["gross_points"] - 2 * BASE_SPREAD_POINTS
events["net_bp_2x"] = events["gross_bp"] - 2 * BASE_SPREAD_POINTS / events["entry"] * 10000.0

def round_stake(x):
    return max(0.01, float(Decimal(str(x)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)))

def max_dd(curve):
    a = np.asarray(curve, dtype=float)
    peak = np.maximum.accumulate(a)
    return float(np.min(a / peak - 1.0))

def summarize(label, frame, spread_mult):
    g = frame.copy().sort_values("trade_date")
    if g.empty:
        print(f"SEALED_Q3_SUMMARY,{label},cost={'BASE' if spread_mult == 1 else 'STRESS2X'},n=0")
        return

    gross = g["gross_bp"].to_numpy(float)
    net = gross - spread_mult * BASE_SPREAD_POINTS / g["entry"].to_numpy(float) * 10000.0
    gross_pts = g["gross_points"].to_numpy(float)

    eq = START_EQUITY
    curve = [eq]
    margins = []
    for _, r in g.iterrows():
        stake = round_stake(eq / float(r["entry"]))
        margins.append(stake * float(r["entry"]) * MARGIN_RATE / eq)
        net_pts = float(r["gross_points"]) - spread_mult * BASE_SPREAD_POINTS
        eq += stake * net_pts
        curve.append(eq)

    print(
        f"SEALED_Q3_SUMMARY,{label},cost={'BASE' if spread_mult == 1 else 'STRESS2X'},"
        f"n={len(g)},"
        f"grossMeanBp={gross.mean():.3f},grossMedianBp={np.median(gross):.3f},"
        f"grossHit={(gross > 0).mean():.3f},"
        f"netMeanBp={net.mean():.3f},netMedianBp={np.median(net):.3f},"
        f"netHit={(net > 0).mean():.3f},"
        f"breakEvenBp={gross.mean():.3f},breakEvenPts={gross_pts.mean():.3f},"
        f"endingEquity={eq:.2f},pnlGBP={eq - START_EQUITY:.2f},"
        f"maxDD={max_dd(curve):.4f},marginMax={max(margins):.4f}"
    )

events = events.sort_values("trade_date").reset_index(drop=True)

for i, r in events.iterrows():
    print(
        f"SEALED_Q3_TRADE,{i+1},"
        f"signal_date={r['signal_date'].date()},trade_date={r['trade_date'].date()},"
        f"z={r['z']:.6f},abs_z={r['abs_z']:.6f},"
        f"signalRetBp={r['signal_return']*10000:.3f},"
        f"fade={'BUY' if r['fade_sign'] > 0 else 'SELL'},"
        f"P95={'yes' if r['p95'] else 'no'},"
        f"entry={r['entry']:.4f},exit={r['exit']:.4f},"
        f"grossPts={r['gross_points']:.3f},grossBp={r['gross_bp']:.3f},"
        f"netBp={r['net_bp']:.3f},netBp2x={r['net_bp_2x']:.3f}"
    )

summarize("P90", events, 1.0)
summarize("P90", events, 2.0)
summarize("P95", events[events["p95"]], 1.0)
summarize("P95", events[events["p95"]], 2.0)

print(
    f"SEALED_Q3_END,p90_n={len(events)},p95_n={int(events['p95'].sum()) if len(events) else 0},"
    "signal_window=2026-07-02_to_2026-09-29"
)

# Also save a compact ledger in the research project workspace.
ledger_cols = [
    "signal_date", "trade_date", "z", "abs_z", "signal_return", "signal_direction", "p95",
    "entry", "exit", "gross_points", "gross_bp", "net_points", "net_bp",
    "net_points_2x", "net_bp_2x"
]
events[ledger_cols].to_csv("SEALED_Q3_LEDGER.csv", index=False)
print("WROTE,SEALED_Q3_LEDGER.csv")
